# **PUC-Rio | ENG 4560 — Projeto Integrado VI: Distribuição Física**
# **Sprint 1 — Limites de tempo e gaps no CVRP**

---

## Objetivo

Este notebook mantém o mesmo modelo inteiro de roteamento usado na Aula 4 e
organiza a análise computacional em quatro partes:

1. resolver a instância com limites de **1 min, 5 min, 10 min, 30 min, 1 h, 2 h e 6 h**;
2. comparar graficamente os resultados dos limites de tempo;
3. resolver a instância com diferentes tolerâncias de gap;
4. comparar graficamente os resultados das tolerâncias de gap.

Em cada execução são registradas as rotas e as principais métricas: custo,
bound, gap observado, tempo, número e percentual de clientes atendidos, demanda
atendida, quantidade de rotas, veículos usados, distância e duração estimada.

Os experimentos são independentes: cada linha da tabela vem de um novo modelo e
de uma nova chamada ao solver.

In [ ]:
# =====================================================
# (1) AMBIENTE
# =====================================================
# Três solvers exatos, para a comparação:
#   CBC    — open-source clássico (referência "sempre disponível")
#   HiGHS  — open-source moderno
#   Gurobi — comercial, licença WLS acadêmica
#
# Sem folium/mapas aqui: este notebook não desenha rotas, só mede desempenho.

# Instala no próprio kernel e funciona em Windows, Linux ou Colab.
%pip install -q pyomo highspy gurobipy pulp

In [ ]:
# =====================================================
# CAMINHOS PADRONIZADOS DA INSTÂNCIA
# =====================================================
INSTANCIA = "C4"

import os
from pathlib import Path

ARQUIVOS = ("nodes.csv", "D.npy", "Cvar.npy", "q.npy", "s.npy",
            "Tmov_h.npy", "params.json")
current_dir = Path.cwd().resolve()

# Funciona quando o Jupyter inicia na raiz, na pasta da instância, em dados ou
# em notebooks. A busca é local e limitada aos ancestrais próximos.
candidate_roots = []
for anchor in (current_dir, *list(current_dir.parents)[:4]):
    candidate_roots.extend((
        anchor if anchor.name.upper() == INSTANCIA else anchor / INSTANCIA,
        anchor / "Projeto_Dis_Fis" / INSTANCIA,
    ))

instance_dir = next(
    (folder for folder in candidate_roots
     if all((folder / "dados" / name).is_file() for name in ARQUIVOS)),
    None,
)
if instance_dir is None:
    attempted = "\n".join(f"- {folder / 'dados'}" for folder in candidate_roots)
    raise FileNotFoundError(
        f"Dados da instância {INSTANCIA} não encontrados. Pastas verificadas:\n{attempted}"
    )

base_dir = instance_dir / "dados"
PASTA_NOTEBOOKS = instance_dir / "notebooks"
PASTA_CONFIG = instance_dir / "configuracao"
PASTA_RESULTADOS = instance_dir / "resultados"
PASTA_TABELAS = PASTA_RESULTADOS / "tabelas"
PASTA_GRAFICOS = PASTA_RESULTADOS / "graficos"
PASTA_LOGS = PASTA_RESULTADOS / "logs"
PASTA_RESUMOS = PASTA_RESULTADOS / "resumos"
PASTA_CHECKPOINTS = PASTA_RESULTADOS / "checkpoints"

for folder in (PASTA_TABELAS, PASTA_GRAFICOS, PASTA_LOGS,
               PASTA_RESUMOS, PASTA_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

cvar_file = base_dir / "Cvar.npy"
license_file = PASTA_CONFIG / "gurobi.lic"
if license_file.is_file():
    for key in ("GRB_WLSACCESSID", "GRB_WLSSECRET", "GRB_LICENSEID"):
        os.environ.pop(key, None)
    os.environ["GRB_LICENSE_FILE"] = str(license_file.resolve())
    print(f"Licença Gurobi: {license_file}")
else:
    print("Licença Gurobi local não encontrada; será usada a licença disponível no ambiente.")

print(f"Instância {INSTANCIA}: {instance_dir}")
print(f"Dados: {base_dir}")
print(f"Resultados: {PASTA_RESULTADOS}")


In [ ]:
# =====================================================
# (3) LEITURA E VALIDAÇÃO
# =====================================================
import numpy as np, pandas as pd, json, math, time, re

try:
    from IPython.display import display as _ipy
except ImportError:
    _ipy = print
def mostrar(o):
    try: _ipy(o)
    except Exception: print(o)

nodes = pd.read_csv(base_dir / "nodes.csv")
D  = np.load(base_dir / "D.npy")
C  = np.load(base_dir / "Cvar.npy")
q  = np.load(base_dir / "q.npy")
s  = np.load(base_dir / "s.npy")
Tmov_h = np.load(base_dir / "Tmov_h.npy")
with (base_dir / "params.json").open(encoding="utf-8") as file:
    params = json.load(file)
n = len(nodes)

assert D.shape == (n, n) and C.shape == (n, n) and Tmov_h.shape == (n, n)
assert q.shape == (n,) and s.shape == (n,)

print(f"Instância {INSTANCIA}: {n-1} clientes + depósito")
print(f"Demanda total: {q[1:].sum():.1f} kg | maior individual: {q[1:].max():.1f} kg")

In [ ]:
# =====================================================
# (4) PARÂMETROS LOGÍSTICOS — FROTA HETEROGÊNEA
# =====================================================
vehicle_types = ["FIO", "VUC"]
Q = {
    "FIO": float(params["VEHICLES"]["Fiorino"]["Q_kg"]),
    "VUC": float(params["VEHICLES"]["VUC"]["Q_kg"]),
}
f = {
    "FIO": float(params["VEHICLES"]["Fiorino"]["custo_fixo_diario"]),
    "VUC": float(params["VEHICLES"]["VUC"]["custo_fixo_diario"]),
}
H, v_kmh = float(params["H_horas"]), float(params["v_kmh"])
T = Tmov_h

Q_BASE, f_BASE, TIPOS_BASE = dict(Q), dict(f), list(vehicle_types)

for k in vehicle_types:
    print(f"  {k}: Q = {Q[k]:6.0f} kg | custo fixo = R$ {f[k]:6.2f}")

In [ ]:
# =====================================================
# (5) MODELO — RESTRIÇÕES IDÊNTICAS ÀS DO NOTEBOOK DA AULA
# =====================================================
# Nada foi acrescentado nem removido em relação às células (5)–(14) da Aula 4.
# A única diferença é estrutural: o modelo é construído por uma FUNÇÃO, para que
# cada experimento possa reconstruí-lo do zero. Isso é obrigatório no Pyomo,
# porque Objective(rule=...) é avaliada uma única vez e guarda os NÚMEROS —
# mudar o dicionário f depois não altera a expressão já construída.
# =====================================================

from pyomo.environ import (ConcreteModel, RangeSet, Set, Var, Binary,
                           NonNegativeReals, Constraint, Objective,
                           minimize, value)
from pyomo.opt import SolverFactory

def build_model(Q=None, f=None, vehicle_types=None, use_mtz=True):
    Qk = {**Q_BASE, **(Q or {})}
    fk = {**f_BASE, **(f or {})}
    KT = list(vehicle_types) if vehicle_types is not None else list(TIPOS_BASE)
    Qk = {k: Qk[k] for k in KT}; fk = {k: fk[k] for k in KT}

    m = ConcreteModel()
    m.N = RangeSet(0, n-1)
    m.C = RangeSet(1, n-1)
    m.K = Set(initialize=KT)
    m.A = [(i,j,k) for i in range(n) for j in range(n) for k in KT if i != j]
    m.x = Var(m.A, domain=Binary)          # (5) arco (i,j) percorrido pelo tipo k
    m.u = Var(m.C, bounds=(1, n-1))        # (6) posição na sequência (MTZ)
    m.y = Var(m.K, domain=Binary)          # (7) tipo de veículo ativado

    # (7) ativação: saiu ou voltou ao depósito => veículo acionado
    m.activate_vehicle = Constraint(
        m.K, rule=lambda m,k: sum(m.x[0,j,k] for j in m.C) <= m.y[k])
    m.activate_return = Constraint(
        m.K, rule=lambda m,k: sum(m.x[i,0,k] for i in m.C) <= m.y[k])

    # (9) custo variável (deslocamento) + custo fixo (frota)
    m.obj = Objective(
        rule=lambda m: (sum(C[i,j]*m.x[i,j,k] for (i,j,k) in m.A)
                        + sum(fk[k]*m.y[k] for k in m.K)), sense=minimize)

    # (10) grau: exatamente uma saída e uma entrada por cliente
    m.out = Constraint(m.C, rule=lambda m,i:
        sum(m.x[i,j,k] for j in m.N if j != i for k in m.K) == 1)
    m.inn = Constraint(m.C, rule=lambda m,j:
        sum(m.x[i,j,k] for i in m.N if i != j for k in m.K) == 1)

    # (11) balanço no depósito e no máximo uma rota por tipo
    m.depot_balance = Constraint(m.K, rule=lambda m,k:
        sum(m.x[0,j,k] for j in m.C) == sum(m.x[i,0,k] for i in m.C))
    m.single_departure = Constraint(m.K, rule=lambda m,k:
        sum(m.x[0,j,k] for j in m.C) <= 1)

    # (12) capacidade agregada heterogênea
    m.capacity = Constraint(rule=lambda m:
        sum(q[i] for i in range(1,n)) <= sum(Qk[k]*m.y[k] for k in m.K))

    # (13) conservação de fluxo por tipo de veículo
    m.flow_by_type = Constraint(m.C, m.K, rule=lambda m,i,k:
        (sum(m.x[i,j,k] for j in m.N if j != i)
         - sum(m.x[j,i,k] for j in m.N if j != i)) == 0)

    # (14) MTZ — eliminação de subtours
    if use_mtz:
        def _mtz(m, i, j):
            if i == j: return Constraint.Skip
            return m.u[i] - m.u[j] + (n-1)*sum(m.x[i,j,k] for k in m.K) <= n-2
        m.mtz = Constraint(m.C, m.C, rule=_mtz)
    return m

_m = build_model()
N_BIN = len(_m.A)
N_CON = len(list(_m.component_data_objects(Constraint)))
print(f"Binárias: {N_BIN} | restrições: {N_CON} | clientes: {n-1}")

In [ ]:
# =====================================================
# (6) SOLVER, ROTAS E MÉTRICAS
# =====================================================
# Esta infraestrutura é compartilhada pelas Partes 1 e 3. Cada cenário constrói
# o modelo inteiro do zero, configura TimeLimit/MIPGap e extrai a solução obtida.

from collections import defaultdict

_OPCOES = {
    "gurobi": {"tl": "TimeLimit", "gap": "MIPGap"},
    "highs":  {"tl": "time_limit", "gap": "mip_rel_gap"},
    "cbc":    {"tl": "seconds", "gap": "ratioGap"},
    "glpk":   {"tl": "tmlim", "gap": "mipgap"},
    "cplex":  {"tl": "timelimit", "gap": "mipgap"},
}

def familia_solver(nome):
    return next((fam for fam in _OPCOES if fam in nome), None)

def criar_solver(nome):
    if nome == "cbc":
        try:
            import pulp
            return SolverFactory("cbc", executable=pulp.PULP_CBC_CMD().path)
        except Exception:
            pass
    return SolverFactory(nome)

def solvers_disponiveis(candidatos=("gurobi_direct", "appsi_highs", "cbc")):
    disponiveis = []
    for nome in candidatos:
        try:
            if criar_solver(nome).available(exception_flag=False):
                disponiveis.append(nome)
        except Exception:
            pass
    return disponiveis

def rotulo_tempo(segundos):
    if segundos < 60:
        return f"{segundos:g}s"
    if segundos < 3600:
        return f"{segundos/60:g}min"
    return f"{segundos/3600:g}h"

def _valor(var):
    try:
        v = value(var, exception=False)
        return float(v) if v is not None else math.nan
    except Exception:
        return math.nan

def arcos_selecionados(mdl):
    return [(i, j, k) for (i, j, k) in mdl.A
            if math.isfinite(_valor(mdl.x[i, j, k])) and _valor(mdl.x[i, j, k]) > 0.5]

def extrair_rotas(mdl):
    """Reconstrói as rotas que saem do depósito a partir dos arcos selecionados."""
    por_tipo = defaultdict(list)
    for i, j, k in arcos_selecionados(mdl):
        por_tipo[k].append((i, j))

    rotas = []
    for k, arcos in por_tipo.items():
        sucessores = defaultdict(list)
        for i, j in arcos:
            sucessores[i].append(j)
        for primeiro in sucessores.get(0, []):
            rota, atual = [0, primeiro], primeiro
            visitados = {(0, primeiro)}
            while atual != 0 and len(rota) <= n + 1:
                candidatos = [j for j in sucessores.get(atual, [])
                              if (atual, j) not in visitados]
                if not candidatos:
                    break
                prox = candidatos[0]
                visitados.add((atual, prox))
                rota.append(prox)
                atual = prox
            rotas.append((k, rota))
    return rotas

def metricas_rotas(mdl):
    rotas = extrair_rotas(mdl)
    clientes = sorted({i for _, rota in rotas for i in rota if i != 0})
    km_total = sum(D[rota[a], rota[a+1]]
                   for _, rota in rotas for a in range(len(rota)-1))
    horas_total = sum(
        sum(T[rota[a], rota[a+1]] for a in range(len(rota)-1))
        + sum(s[i] for i in rota if i != 0)
        for _, rota in rotas
    )
    veiculos = sorted({k for k, _ in rotas})
    texto = " | ".join(f"[{k}] " + "-".join(map(str, rota)) for k, rota in rotas)
    return {
        "qtd_clientes_atendidos": len(clientes),
        "pct_clientes_atendidos": 100 * len(clientes) / (n-1),
        "demanda_atendida_kg": float(q[clientes].sum()) if clientes else 0.0,
        "qtd_rotas": len(rotas),
        "qtd_veiculos": len(veiculos),
        "veiculos": ", ".join(veiculos),
        "distancia_total_km": float(km_total),
        "tempo_total_rotas_h": float(horas_total),
        "rotas": texto,
    }

def resolver_cenario(time_limit, mip_gap=None):
    """Resolve um cenário limitado e devolve modelo e métricas padronizadas."""
    if time_limit is None or time_limit <= 0:
        raise ValueError("time_limit deve ser positivo.")
    mdl = build_model()
    solver = criar_solver(SOLVER_PADRAO)
    fam = familia_solver(SOLVER_PADRAO)
    solver.options[_OPCOES[fam]["tl"]] = float(time_limit)
    if mip_gap is not None:
        solver.options[_OPCOES[fam]["gap"]] = float(mip_gap)

    inicio = time.time()
    resultado = solver.solve(mdl, tee=False, load_solutions=False)
    tempo_s = time.time() - inicio

    carregou = False
    try:
        mdl.solutions.load_from(resultado)
        carregou = True
    except Exception:
        pass

    custo = _valor(mdl.obj) if carregou else math.nan
    try:
        bound = float(resultado.problem.lower_bound)
        if not math.isfinite(bound):
            bound = math.nan
    except Exception:
        bound = math.nan
    gap_pct = (100 * abs(custo - bound) / abs(custo)
               if math.isfinite(custo) and math.isfinite(bound) and abs(custo) > 1e-9
               else math.nan)
    termination = str(resultado.solver.termination_condition)

    metricas = {
        "solver": SOLVER_PADRAO,
        "time_limit_s": time_limit,
        "mip_gap_pedido_pct": (100*mip_gap if mip_gap is not None else math.nan),
        "tempo_s": tempo_s,
        "status": str(resultado.solver.status),
        "termination": termination,
        "custo": custo,
        "bound": bound,
        "gap_real_pct": gap_pct,
        "otimo_provado": bool(math.isfinite(gap_pct) and gap_pct <= 1e-6),
        "tem_solucao": carregou and math.isfinite(custo),
    }
    metricas.update(metricas_rotas(mdl) if metricas["tem_solucao"] else {
        "qtd_clientes_atendidos": 0, "pct_clientes_atendidos": 0.0,
        "demanda_atendida_kg": 0.0, "qtd_rotas": 0, "qtd_veiculos": 0,
        "veiculos": "", "distancia_total_km": math.nan,
        "tempo_total_rotas_h": math.nan, "rotas": "",
    })
    return mdl, metricas

DISPONIVEIS = solvers_disponiveis()
if not DISPONIVEIS:
    raise RuntimeError("Nenhum solver disponível — execute a célula de ambiente.")
SOLVER_PADRAO = DISPONIVEIS[0]
print("Solvers disponíveis:", DISPONIVEIS)
print("Solver usado nos dois experimentos:", SOLVER_PADRAO)

---
# **PARTE 1 — Cálculo das rotas com diferentes limites de tempo**

Cada limite inicia uma otimização independente. A tabela e o CSV incluem tanto
as métricas computacionais quanto as métricas operacionais e a sequência de cada
rota encontrada.

In [ ]:
# =====================================================
# PARTE 1 — TIME LIMITS: EXECUÇÃO E MÉTRICAS
# =====================================================
TIME_LIMITS = [60, 300, 600, 1800, 3600, 7200, 21600]
#               1m   5m   10m   30m    1h    2h     6h

print(f"Instância: {INSTANCIA} | clientes: {n-1} | solver: {SOLVER_PADRAO}")
print("Limites:", [rotulo_tempo(t) for t in TIME_LIMITS])
print("Pior caso para esta parte:", rotulo_tempo(sum(TIME_LIMITS)))

registros_tempo = []
modelos_tempo = {}
for tl in TIME_LIMITS:
    print(f"\nRodando TimeLimit = {rotulo_tempo(tl)}...", flush=True)
    mdl, met = resolver_cenario(time_limit=tl)
    met["limite"] = rotulo_tempo(tl)
    registros_tempo.append(met)
    modelos_tempo[tl] = mdl
    print(f"  {met['termination']} | custo R$ {met['custo']:.2f} | "
          f"gap {met['gap_real_pct']:.3f}% | clientes "
          f"{met['qtd_clientes_atendidos']}/{n-1} | rotas {met['qtd_rotas']}")

df_tempo = pd.DataFrame(registros_tempo)
custos_validos = df_tempo.loc[df_tempo["custo"].notna(), "custo"]
melhor_custo_tempo = custos_validos.min() if len(custos_validos) else math.nan
df_tempo["custo_acima_melhor_pct"] = (
    100 * (df_tempo["custo"] - melhor_custo_tempo) / melhor_custo_tempo
    if math.isfinite(melhor_custo_tempo) else math.nan
)
arquivo_tempo = PASTA_TABELAS / f"P1_time_limits_{INSTANCIA}.csv"
df_tempo.to_csv(arquivo_tempo, index=False)

colunas = ["limite", "tempo_s", "custo", "bound", "gap_real_pct",
           "qtd_clientes_atendidos", "pct_clientes_atendidos", "qtd_rotas",
           "qtd_veiculos", "distancia_total_km", "demanda_atendida_kg",
           "termination"]
mostrar(df_tempo[colunas].round(3))
print("CSV salvo em:", arquivo_tempo)

---
# **PARTE 2 — Gráficos dos limites de tempo**

Os painéis mostram custo, gap observado, atendimento e estrutura operacional da
solução. Assim, uma execução sem solução viável ou com atendimento incompleto
fica visível, em vez de ser avaliada apenas pelo custo.

In [ ]:
# =====================================================
# PARTE 2 — TIME LIMITS: GRÁFICOS
# =====================================================
import matplotlib.pyplot as plt

d = df_tempo.copy()
x = np.arange(len(d))
labels = d["limite"].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(x, d["custo"], "o-", lw=2, color="#1f6feb", label="Custo")
ax.plot(x, d["bound"], "s--", lw=1.8, color="#d1242f", label="Bound")
ax.set_ylabel("R$"); ax.set_title("(a) Custo e bound")
ax.legend(); ax.grid(alpha=.3)

ax = axes[0, 1]
ax.plot(x, d["gap_real_pct"], "o-", lw=2, color="#8250df")
ax.axhline(0, color="#0b8a3e", ls=":")
ax.set_ylabel("Gap (%)"); ax.set_title("(b) Gap observado")
ax.grid(alpha=.3)

ax = axes[1, 0]
ax.bar(x, d["qtd_clientes_atendidos"], color="#0b8a3e")
ax.axhline(n-1, color="#333", ls="--", label=f"Total = {n-1}")
ax.set_ylabel("Clientes"); ax.set_title("(c) Clientes atendidos")
ax.legend(); ax.grid(alpha=.3, axis="y")

ax = axes[1, 1]
ax.plot(x, d["distancia_total_km"], "o-", color="#bf8700", lw=2, label="Distância (km)")
ax2 = ax.twinx()
ax2.plot(x, d["qtd_rotas"], "s--", color="#d1242f", lw=1.8, label="Rotas")
ax.set_ylabel("Distância (km)"); ax2.set_ylabel("Quantidade de rotas")
ax.set_title("(d) Distância e rotas"); ax.grid(alpha=.3)

for ax in axes.flat:
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_xlabel("Time limit")

fig.suptitle(f"Limites de tempo — {INSTANCIA} | {SOLVER_PADRAO}", fontweight="bold")
fig.tight_layout()
arquivo_fig_tempo = PASTA_GRAFICOS / f"P2_time_limits_{INSTANCIA}.png"
fig.savefig(arquivo_fig_tempo, dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva em:", arquivo_fig_tempo)

---
# **PARTE 3 — Cálculo das rotas com diferentes gaps**

Cada tolerância de gap inicia uma otimização independente. O limite de 6 horas é
somente uma proteção máxima por rodada; o solver pode encerrar antes ao atingir
a tolerância pedida.

In [ ]:
# =====================================================
# PARTE 3 — GAPS: EXECUÇÃO E MÉTRICAS
# =====================================================
GAPS = [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.0]
#        20%   10%    5%    2%    1%    0,5%   0%
TIME_LIMIT_GAPS = 21600  # 6 h de proteção por execução

print(f"Instância: {INSTANCIA} | solver: {SOLVER_PADRAO}")
print("Gaps pedidos:", [f"{100*g:g}%" for g in GAPS])
print("Pior caso para esta parte:", rotulo_tempo(TIME_LIMIT_GAPS * len(GAPS)))

registros_gap = []
modelos_gap = {}
for gap in GAPS:
    print(f"\nRodando MIPGap = {100*gap:g}%...", flush=True)
    mdl, met = resolver_cenario(time_limit=TIME_LIMIT_GAPS, mip_gap=gap)
    met["gap_pedido"] = f"{100*gap:g}%"
    registros_gap.append(met)
    modelos_gap[gap] = mdl
    print(f"  {met['termination']} | tempo {met['tempo_s']:.1f}s | "
          f"custo R$ {met['custo']:.2f} | gap real {met['gap_real_pct']:.3f}% | "
          f"clientes {met['qtd_clientes_atendidos']}/{n-1} | rotas {met['qtd_rotas']}")

df_gap = pd.DataFrame(registros_gap)
custos_validos = df_gap.loc[df_gap["custo"].notna(), "custo"]
melhor_custo_gap = custos_validos.min() if len(custos_validos) else math.nan
df_gap["custo_acima_melhor_pct"] = (
    100 * (df_gap["custo"] - melhor_custo_gap) / melhor_custo_gap
    if math.isfinite(melhor_custo_gap) else math.nan
)
arquivo_gap = PASTA_TABELAS / f"P3_gaps_{INSTANCIA}.csv"
df_gap.to_csv(arquivo_gap, index=False)

colunas = ["gap_pedido", "tempo_s", "custo", "bound", "gap_real_pct",
           "custo_acima_melhor_pct", "qtd_clientes_atendidos",
           "pct_clientes_atendidos", "qtd_rotas", "qtd_veiculos",
           "distancia_total_km", "demanda_atendida_kg", "termination"]
mostrar(df_gap[colunas].round(3))
print("CSV salvo em:", arquivo_gap)

---
# **PARTE 4 — Gráficos dos diferentes gaps**

Os gráficos permitem comparar o ganho de tempo ao aceitar uma tolerância maior
com o custo e a configuração operacional das rotas devolvidas.

In [ ]:
# =====================================================
# PARTE 4 — GAPS: GRÁFICOS
# =====================================================
d = df_gap.sort_values("mip_gap_pedido_pct", ascending=False).reset_index(drop=True)
x = np.arange(len(d))
labels = d["gap_pedido"].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.bar(x, d["tempo_s"], color="#1f6feb")
ax.set_ylabel("Tempo efetivo (s)"); ax.set_title("(a) Esforço computacional")
ax.grid(alpha=.3, axis="y")

ax = axes[0, 1]
ax.plot(x, d["custo"], "o-", lw=2, color="#bf8700")
ax.set_ylabel("Custo (R$)"); ax.set_title("(b) Custo da solução")
ax.grid(alpha=.3)

ax = axes[1, 0]
ax.bar(x, d["qtd_clientes_atendidos"], color="#0b8a3e")
ax.axhline(n-1, color="#333", ls="--", label=f"Total = {n-1}")
ax.set_ylabel("Clientes"); ax.set_title("(c) Clientes atendidos")
ax.legend(); ax.grid(alpha=.3, axis="y")

ax = axes[1, 1]
ax.plot(x, d["gap_real_pct"], "o-", color="#8250df", lw=2, label="Gap real")
ax.plot(x, d["mip_gap_pedido_pct"], "s--", color="#d1242f", lw=1.8, label="Gap pedido")
ax.set_ylabel("Gap (%)"); ax.set_title("(d) Gap pedido × observado")
ax.legend(); ax.grid(alpha=.3)

for ax in axes.flat:
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_xlabel("Tolerância de gap")

fig.suptitle(f"Tolerâncias de gap — {INSTANCIA} | {SOLVER_PADRAO}", fontweight="bold")
fig.tight_layout()
arquivo_fig_gap = PASTA_GRAFICOS / f"P4_gaps_{INSTANCIA}.png"
fig.savefig(arquivo_fig_gap, dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva em:", arquivo_fig_gap)

print("\nArquivos desta análise:")
for arquivo in (arquivo_tempo, arquivo_fig_tempo, arquivo_gap, arquivo_fig_gap):
    print(" ", arquivo)